In [ ]:
import pandas as pd

 Сколько отсекает каждый фильтр:

In [2]:
predf = pd.read_excel("../../data/interim/preclean_base.xlsx")

df = predf.copy()

conditions = {
    "birthday_mistake_ok": (df["birthday_mistake"].isna() | (df["birthday_mistake"] == 0)),
    "busy_type_full": (df["busy_type"] == "Полная занятость"),
    "country_rf": (df["country"] == "Российская Федерация"),
    "inner_info_status_approved": (df["inner_info_status"] == "Одобрено"),
    "experience_mistake_ok": (df["experience_mistake"] == 0),
    "inactive_eq_1": (df["inactive"] == 1),
    "relocation_ok": (df["relocation"].isna() | (df["relocation"] == 0)),
    "retraining_ok": (df["retraining_capability"].isna() | (df["retraining_capability"] == 1)),
    "business_trips_eq_0": (df["business_trips"] == 0),
    "gender_notna": df["gender"].notna(),
    "gender_not_empty": (df["gender"] != ""),
    "salary_gt_1000": (df["salary"] > 1000),
}

n_total = len(df)

rows = []
for name, cond in conditions.items():
    kept = int(cond.sum())
    dropped = n_total - kept
    rows.append({
        "filter": name,
        "kept_rows": kept,
        "dropped_rows": dropped,
        "dropped_pct": round(dropped / n_total * 100, 2)
    })

filter_impact_df = pd.DataFrame(rows).sort_values("dropped_rows", ascending=False)
print(filter_impact_df)

                        filter  kept_rows  dropped_rows  dropped_pct
9                 gender_notna      35048         14951        29.90
1               busy_type_full      46532          3467         6.93
8          business_trips_eq_0      46716          3283         6.57
5                inactive_eq_1      48394          1605         3.21
3   inner_info_status_approved      49467           532         1.06
6                relocation_ok      49514           485         0.97
11              salary_gt_1000      49576           423         0.85
7                retraining_ok      49680           319         0.64
2                   country_rf      49731           268         0.54
0          birthday_mistake_ok      49995             4         0.01
4        experience_mistake_ok      49995             4         0.01
10            gender_not_empty      49999             0         0.00


In [3]:
mask = (
    (df["birthday_mistake"].isna() | (df["birthday_mistake"] == 0)) &
    (df["busy_type"] == "Полная занятость") &
    (df["country"] == "Российская Федерация") &
    (df["inner_info_status"] == "Одобрено") &
    (df["experience_mistake"] == 0) &
    (df["inactive"] == 1) &
    (df["relocation"].isna() | df["relocation"] == 0) &
    (df["retraining_capability"].isna() | df["retraining_capability"] == 1) &
    (df["business_trips"] == 0) &
    (df["gender"].notna()) &
    (df["gender"] != "") &
    (df["salary"] > 1000)
)

final_cols = [
    "salary",
    "gender",
    "experience",
    "education_type",
    "region_code",
    "industry_code"
]

tmp_df = df.loc[mask, final_cols].copy()

print("После mask:", tmp_df.shape)

print("\nПропуски после mask:")
print(tmp_df.isna().sum())

print("\nСтроки без пропусков в final_cols:")
print(tmp_df.dropna(subset=final_cols).shape)

После mask: (28834, 6)

Пропуски после mask:
salary                0
gender                0
experience            0
education_type    22404
region_code           0
industry_code      3463
dtype: int64

Строки без пропусков в final_cols:
(5826, 6)
